In [ ]:
# ============================================================
# CELL 1: IMPORT LIBRARIES
# ============================================================

import math
import copy
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [ ]:
# ============================================================
# CELL 2: DEVICE AND REPRODUCIBILITY
# ============================================================

torch.manual_seed(42)
np.random.seed(42)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

In [ ]:
# ============================================================
# CELL 3: LOAD MNIST
# ============================================================

transform = transforms.Compose([

    transforms.ToTensor(),

    # Original ToTensor range:
    #
    #     [0, 1]
    #
    # Convert to:
    #
    #     [-1, 1]

    transforms.Normalize(
        mean=(0.5,),
        std=(0.5,)
    )
])


train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    transform=transform,
    download=True
)


test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    transform=transform,
    download=True
)


BATCH_SIZE = 128


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)


test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


print("Training images:", len(train_dataset))
print("Test images:", len(test_dataset))

In [ ]:
# ============================================================
# CELL 4: VISUALIZE MNIST
# ============================================================

images, labels = next(iter(train_loader))

# Undo normalization:
#
# [-1,1] -> [0,1]

show_images = (
    images + 1
) / 2


plt.figure(figsize=(10, 4))

for i in range(10):

    plt.subplot(2, 5, i + 1)

    plt.imshow(
        show_images[i].squeeze(),
        cmap="gray"
    )

    plt.title(
        str(labels[i].item())
    )

    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 5: DIFFUSION NOISE SCHEDULE
# ============================================================

# Number of diffusion steps.
T = 500


# ------------------------------------------------------------
# beta_t = 1 - alpha_t
#
# Therefore:
#
# alpha_t = 1 - beta_t
#
# Paper Eq. (31):
#
# q(x_t | x_{t-1})
#
# = N(
#       sqrt(alpha_t) x_{t-1},
#       (1-alpha_t) I
#     )
#
# Since:
#
# beta_t = 1-alpha_t
#
# variance at one transition = beta_t.
# ------------------------------------------------------------

beta_start = 1e-4
beta_end   = 0.03


betas = torch.linspace(
    beta_start,
    beta_end,
    T,
    device=device
)


# alpha_t
alphas = 1.0 - betas


# ------------------------------------------------------------
# alpha_bar_t
#
#     alpha_bar_t
#
#       = product_{s=1}^t alpha_s
#
# Used throughout Eq. (69), Eq. (70), Eq. (85),
# Eq. (93), Eq. (115), Eq. (125), Eq. (143), etc.
# ------------------------------------------------------------

alpha_bars = torch.cumprod(
    alphas,
    dim=0
)


# We frequently need alpha_bar_{t-1}.
#
# At t=1:
#
# alpha_bar_0 = 1.
#
# Therefore prepend 1.

alpha_bars_prev = torch.cat([

    torch.ones(
        1,
        device=device
    ),

    alpha_bars[:-1]

])


print("alpha_bar_T =",
      alpha_bars[-1].item())

In [ ]:
# ============================================================
# CELL 6: EXTRACT TIMESTEP-SPECIFIC COEFFICIENT
# ============================================================

def extract(values, t, x_shape):
    """
    values:
        Tensor containing one value for each timestep.

        Shape:
            [T]

    t:
        Timesteps for current batch.

        IMPORTANT:
        In our code t is 0-indexed:

            t = 0 corresponds to paper timestep 1
            t = 1 corresponds to paper timestep 2
            ...
            t = T-1 corresponds to paper timestep T

    Returns:
        Shape [B,1,1,1]

    This lets PyTorch broadcast the coefficient over
    all image pixels.
    """

    batch_size = t.shape[0]

    out = values.gather(
        0,
        t
    )

    return out.reshape(
        batch_size,
        *((1,) * (len(x_shape) - 1))
    )

In [ ]:
# ============================================================
# CELL 7: SAMPLE x_t DIRECTLY FROM x_0
# ============================================================

def q_sample(x0, t, noise=None):
    """
    Sample x_t directly from x_0.

    ----------------------------------------------------------
    PAPER EQUATION (69)
    ----------------------------------------------------------

        x_t
        =
        sqrt(alpha_bar_t) * x_0
        +
        sqrt(1 - alpha_bar_t) * epsilon

        epsilon ~ N(0, I)

    Equivalent distribution, paper Eq. (70):

        q(x_t | x_0)

        = N(
            sqrt(alpha_bar_t) x_0,
            (1-alpha_bar_t) I
          )

    ----------------------------------------------------------
    """

    if noise is None:

        # epsilon ~ N(0,I)
        noise = torch.randn_like(x0)


    alpha_bar_t = extract(
        alpha_bars,
        t,
        x0.shape
    )


    # ========================================================
    # IMPLEMENTING PAPER EQUATION (69)
    # ========================================================

    xt = (

        torch.sqrt(
            alpha_bar_t
        ) * x0

        +

        torch.sqrt(
            1.0 - alpha_bar_t
        ) * noise

    )


    return xt, noise

In [ ]:
# ============================================================
# CELL 8: VISUALIZE FORWARD DIFFUSION
# ============================================================

x0 = images[0:1].to(device)


paper_timesteps = [
    1,
    50,
    100,
    150,
    200,
    250,
    300,
    350,
    400,
    450,
    500
]


plt.figure(figsize=(14, 2))


for i, paper_t in enumerate(paper_timesteps):

    # Our Python arrays are zero indexed.
    t = torch.tensor(
        [paper_t - 1],
        device=device
    )

    xt, epsilon = q_sample(
        x0,
        t
    )


    xt_display = (
        xt.clamp(-1, 1) + 1
    ) / 2


    plt.subplot(
        1,
        len(paper_timesteps),
        i + 1
    )

    plt.imshow(
        xt_display[
            0, 0
        ].cpu(),
        cmap="gray"
    )

    plt.title(
        f"t={paper_t}"
    )

    plt.axis("off")


plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 9: POSTERIOR VARIANCE
# ============================================================

# ------------------------------------------------------------
# PAPER EQUATION (85)
#
# sigma_q^2(t)
#
# =
#
# (1-alpha_t)(1-alpha_bar_{t-1})
# --------------------------------
#         1-alpha_bar_t
#
# Since:
#
# beta_t = 1-alpha_t
#
# we implement:
#
# beta_t * (1-alpha_bar_prev)
# --------------------------------
#       1-alpha_bar_t
# ------------------------------------------------------------

posterior_variance = (

    betas

    *

    (
        1.0
        -
        alpha_bars_prev
    )

    /

    (
        1.0
        -
        alpha_bars
    )
)


# t=1 gives zero posterior variance.
#
# For numerical safety when a log is needed:

posterior_log_variance_clipped = torch.log(
    posterior_variance.clamp(
        min=1e-20
    )
)

In [ ]:
# ============================================================
# CELL 10: TRUE POSTERIOR MEAN
# ============================================================

def true_posterior_mean(x0, xt, t):
    """
    Compute the exact q(x_{t-1}|x_t,x_0) posterior mean.

    ----------------------------------------------------------
    PAPER EQUATION (93)
    ----------------------------------------------------------

                        sqrt(alpha_t)
              * (1-alpha_bar_{t-1}) * x_t

                           +

                   sqrt(alpha_bar_{t-1})
                      * (1-alpha_t) * x_0

        mu_q = -----------------------------------------
                         1-alpha_bar_t

    ----------------------------------------------------------
    """

    alpha_t = extract(
        alphas,
        t,
        xt.shape
    )

    alpha_bar_t = extract(
        alpha_bars,
        t,
        xt.shape
    )

    alpha_bar_prev = extract(
        alpha_bars_prev,
        t,
        xt.shape
    )


    # ========================================================
    # PAPER EQUATION (93)
    # ========================================================

    mu_q = (

        torch.sqrt(alpha_t)

        *

        (
            1.0
            -
            alpha_bar_prev
        )

        *

        xt

        +

        torch.sqrt(
            alpha_bar_prev
        )

        *

        (
            1.0
            -
            alpha_t
        )

        *

        x0

    ) / (

        1.0
        -
        alpha_bar_t

    )


    return mu_q

In [ ]:
# ============================================================
# CELL 11: SINUSOIDAL TIME EMBEDDING
# ============================================================

class SinusoidalTimeEmbedding(nn.Module):

    def __init__(self, dim):

        super().__init__()

        self.dim = dim


    def forward(self, t):
        """
        Convert scalar timestep t into a vector.

        Input:
            t shape [B]

        Output:
            embedding shape [B, dim]
        """

        half_dim = self.dim // 2


        frequencies = torch.exp(

            -math.log(10000)

            *

            torch.arange(
                half_dim,
                device=t.device
            )

            /

            (
                half_dim - 1
            )
        )


        angles = (
            t.float()[:, None]
            *
            frequencies[None, :]
        )


        embedding = torch.cat([

            torch.sin(angles),

            torch.cos(angles)

        ], dim=1)


        return embedding

In [ ]:
# ============================================================
# CELL 12: TIME-CONDITIONED RESIDUAL BLOCK
# ============================================================

class ResBlock(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels,
        time_dim
    ):

        super().__init__()


        self.conv1 = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=3,
            padding=1
        )


        self.norm1 = nn.GroupNorm(
            8,
            out_channels
        )


        # Convert timestep embedding into a feature bias.
        self.time_projection = nn.Linear(
            time_dim,
            out_channels
        )


        self.conv2 = nn.Conv2d(
            out_channels,
            out_channels,
            kernel_size=3,
            padding=1
        )


        self.norm2 = nn.GroupNorm(
            8,
            out_channels
        )


        if in_channels != out_channels:

            self.residual = nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=1
            )

        else:

            self.residual = nn.Identity()


    def forward(self, x, time_embedding):

        residual = self.residual(x)


        h = self.conv1(x)

        h = self.norm1(h)

        h = F.silu(h)


        # Add timestep information.
        #
        # time_embedding:
        #     [B, time_dim]
        #
        # projected:
        #     [B, C]
        #
        # reshape:
        #     [B,C,1,1]

        time_information = self.time_projection(
            time_embedding
        )[:, :, None, None]


        h = h + time_information


        h = self.conv2(h)

        h = self.norm2(h)

        h = F.silu(h)


        return h + residual

In [ ]:
# ============================================================
# CELL 13: MNIST TIME-CONDITIONED U-NET
# ============================================================

class MNISTDiffusionUNet(nn.Module):

    def __init__(
        self,
        time_dim=128
    ):

        super().__init__()


        # ----------------------------------------------------
        # TIME NETWORK
        # ----------------------------------------------------

        self.time_embedding = nn.Sequential(

            SinusoidalTimeEmbedding(
                time_dim
            ),

            nn.Linear(
                time_dim,
                time_dim
            ),

            nn.SiLU(),

            nn.Linear(
                time_dim,
                time_dim
            )

        )


        # ----------------------------------------------------
        # INPUT
        #
        # x_t:
        #
        # [B,1,28,28]
        # ----------------------------------------------------

        self.input_conv = nn.Conv2d(
            1,
            64,
            kernel_size=3,
            padding=1
        )


        # ----------------------------------------------------
        # 28 x 28
        # ----------------------------------------------------

        self.enc1 = ResBlock(
            64,
            64,
            time_dim
        )


        # 28 -> 14
        self.down1 = nn.Conv2d(
            64,
            128,
            kernel_size=4,
            stride=2,
            padding=1
        )


        # ----------------------------------------------------
        # 14 x 14
        # ----------------------------------------------------

        self.enc2 = ResBlock(
            128,
            128,
            time_dim
        )


        # 14 -> 7
        self.down2 = nn.Conv2d(
            128,
            256,
            kernel_size=4,
            stride=2,
            padding=1
        )


        # ----------------------------------------------------
        # BOTTLENECK: 7 x 7
        # ----------------------------------------------------

        self.middle1 = ResBlock(
            256,
            256,
            time_dim
        )

        self.middle2 = ResBlock(
            256,
            256,
            time_dim
        )


        # ----------------------------------------------------
        # 7 -> 14
        # ----------------------------------------------------

        self.up1 = nn.ConvTranspose2d(
            256,
            128,
            kernel_size=4,
            stride=2,
            padding=1
        )


        # Concatenate skip:
        #
        # 128 + 128 = 256

        self.dec1 = ResBlock(
            256,
            128,
            time_dim
        )


        # ----------------------------------------------------
        # 14 -> 28
        # ----------------------------------------------------

        self.up2 = nn.ConvTranspose2d(
            128,
            64,
            kernel_size=4,
            stride=2,
            padding=1
        )


        # 64 + 64 skip = 128

        self.dec2 = ResBlock(
            128,
            64,
            time_dim
        )


        # ----------------------------------------------------
        # OUTPUT
        #
        # Always [B,1,28,28].
        #
        # Interpretation depends on the model:
        #
        # mean model  -> mu_theta
        # x0 model    -> x_hat_theta
        # noise model -> epsilon_hat_theta
        # score model -> s_theta
        #
        # NO activation here.
        # ----------------------------------------------------

        self.output = nn.Conv2d(
            64,
            1,
            kernel_size=1
        )


    def forward(self, xt, t):

        # f_theta(x_t,t)

        temb = self.time_embedding(t)


        x1 = self.input_conv(xt)

        x1 = self.enc1(
            x1,
            temb
        )


        x2 = self.down1(x1)

        x2 = self.enc2(
            x2,
            temb
        )


        h = self.down2(x2)

        h = self.middle1(
            h,
            temb
        )

        h = self.middle2(
            h,
            temb
        )


        h = self.up1(h)


        # Skip connection
        h = torch.cat(
            [h, x2],
            dim=1
        )

        h = self.dec1(
            h,
            temb
        )


        h = self.up2(h)


        h = torch.cat(
            [h, x1],
            dim=1
        )

        h = self.dec2(
            h,
            temb
        )


        return self.output(h)

In [ ]:
# ============================================================
# CELL 14: TEST NETWORK
# ============================================================

test_model = MNISTDiffusionUNet().to(device)

x_test = torch.randn(
    8,
    1,
    28,
    28,
    device=device
)

t_test = torch.randint(
    0,
    T,
    (8,),
    device=device
)

y_test = test_model(
    x_test,
    t_test
)

print(
    "Input shape :",
    x_test.shape
)

print(
    "Output shape:",
    y_test.shape
)


del test_model

In [ ]:
# ============================================================
# CELL 15: MODEL 1 -- MEAN TARGET
# ============================================================

def target_mean(x0, xt, t):
    """
    TARGET FOR MODEL 1.

    Network:

        mu_theta(x_t,t)

    Target:

        mu_q(x_t,x_0)

    PAPER EQUATION (93):

                       sqrt(alpha_t)
             (1-alpha_bar_{t-1}) x_t

                           +

                 sqrt(alpha_bar_{t-1})
                    (1-alpha_t) x_0

        mu_q = --------------------------------
                       1-alpha_bar_t
    """

    return true_posterior_mean(
        x0,
        xt,
        t
    )

In [ ]:
# ============================================================
# CELL 16: MODEL 1 -- MEAN LOSS
# ============================================================

def mean_prediction_loss(
    prediction,
    target,
    t,
    exact_paper_weight=True
):
    """
    ----------------------------------------------------------
    PAPER EQUATION (92)
    ----------------------------------------------------------

        argmin_theta

             1
        -------------
        2 sigma_q^2(t)

        * || mu_theta - mu_q ||_2^2


    If exact_paper_weight=True:
        implement the timestep-dependent weighting.

    If False:
        use simple MSE.

    The paper's expression uses ||.||_2^2.
    Here we take the MEAN over image pixels rather than SUM.

    Since image dimensionality is constant, this differs only
    by a positive constant factor and therefore has the same
    minimizer.
    ----------------------------------------------------------
    """

    squared_error = (

        prediction - target

    ).pow(2).flatten(1).mean(1)


    if not exact_paper_weight:

        return squared_error.mean()


    sigma_q_sq = extract(
        posterior_variance,
        t,
        prediction.shape
    ).reshape(
        prediction.shape[0]
    )


    # ========================================================
    # IMPLEMENTING PAPER EQ. (92)
    #
    # 1 / (2 sigma_q^2(t))
    # ========================================================

    weight = (
        1.0
        /
        (
            2.0
            *
            sigma_q_sq
        )
    )


    return (
        weight
        *
        squared_error
    ).mean()

In [ ]:
# ============================================================
# CELL 17: MODEL 2 -- x_0 PREDICTION -> mu_theta
# ============================================================

def x0_prediction_to_mean(
    xt,
    x0_prediction,
    t
):
    """
    ----------------------------------------------------------
    PAPER EQUATION (94)
    ----------------------------------------------------------

                    sqrt(alpha_t)
              (1-alpha_bar_{t-1}) x_t

                         +

                sqrt(alpha_bar_{t-1})
                  (1-alpha_t) x_hat_theta

        mu_theta = ------------------------------
                       1-alpha_bar_t
    ----------------------------------------------------------
    """

    alpha_t = extract(
        alphas,
        t,
        xt.shape
    )

    alpha_bar_t = extract(
        alpha_bars,
        t,
        xt.shape
    )

    alpha_bar_prev = extract(
        alpha_bars_prev,
        t,
        xt.shape
    )


    # ========================================================
    # IMPLEMENT PAPER EQUATION (94)
    # ========================================================

    mu_theta = (

        torch.sqrt(alpha_t)

        *

        (
            1.0
            -
            alpha_bar_prev
        )

        *

        xt

        +

        torch.sqrt(
            alpha_bar_prev
        )

        *

        (
            1.0
            -
            alpha_t
        )

        *

        x0_prediction

    ) / (

        1.0
        -
        alpha_bar_t

    )


    return mu_theta

In [ ]:
# ============================================================
# CELL 18: MODEL 2 -- SOURCE SAMPLE x_0 LOSS
# ============================================================

def x0_prediction_loss(
    prediction,
    x0,
    t,
    exact_paper_weight=True
):

    squared_error = (

        prediction - x0

    ).pow(2).flatten(1).mean(1)


    if not exact_paper_weight:

        return squared_error.mean()


    alpha_t = extract(
        alphas,
        t,
        prediction.shape
    ).reshape(
        prediction.shape[0]
    )


    alpha_bar_t = extract(
        alpha_bars,
        t,
        prediction.shape
    ).reshape(
        prediction.shape[0]
    )


    alpha_bar_prev = extract(
        alpha_bars_prev,
        t,
        prediction.shape
    ).reshape(
        prediction.shape[0]
    )


    sigma_q_sq = extract(
        posterior_variance,
        t,
        prediction.shape
    ).reshape(
        prediction.shape[0]
    )


    # ========================================================
    # IMPLEMENTING PAPER EQUATION (99)
    #
    # 1 / (2 sigma_q^2(t))
    #
    # *
    #
    # alpha_bar_{t-1} (1-alpha_t)^2
    # --------------------------------
    #        (1-alpha_bar_t)^2
    #
    # *
    #
    # || x_hat_theta - x_0 ||^2
    # ========================================================

    weight = (

        1.0

        /

        (
            2.0
            *
            sigma_q_sq
        )

        *

        alpha_bar_prev

        *

        (
            1.0
            -
            alpha_t
        ).pow(2)

        /

        (
            1.0
            -
            alpha_bar_t
        ).pow(2)

    )


    return (
        weight
        *
        squared_error
    ).mean()

In [ ]:
# ============================================================
# CELL 19: NOISE PREDICTION -> x_0
# ============================================================

def epsilon_to_x0(
    xt,
    epsilon_prediction,
    t
):
    """
    ----------------------------------------------------------
    PAPER EQUATION (115)
    ----------------------------------------------------------

                  x_t
                   -
          sqrt(1-alpha_bar_t) epsilon

        x_0 = ---------------------------
                sqrt(alpha_bar_t)

    During inference epsilon is unknown, so replace it with

        epsilon_hat_theta(x_t,t)
    ----------------------------------------------------------
    """

    alpha_bar_t = extract(
        alpha_bars,
        t,
        xt.shape
    )


    # ========================================================
    # IMPLEMENT PAPER EQUATION (115)
    # ========================================================

    x0_prediction = (

        xt

        -

        torch.sqrt(
            1.0
            -
            alpha_bar_t
        )

        *

        epsilon_prediction

    ) / torch.sqrt(
        alpha_bar_t
    )


    return x0_prediction

In [ ]:
# ============================================================
# CELL 20: MODEL 3 -- epsilon_hat -> mu_theta
# ============================================================

def epsilon_prediction_to_mean(
    xt,
    epsilon_prediction,
    t
):
    """
    ----------------------------------------------------------
    PAPER EQUATION (125)
    ----------------------------------------------------------

        mu_theta(x_t,t)

                  1
        = ---------------- x_t
             sqrt(alpha_t)

          -

             (1-alpha_t)
        -------------------------
        sqrt(alpha_t)
        sqrt(1-alpha_bar_t)

          * epsilon_hat_theta
    ----------------------------------------------------------
    """

    alpha_t = extract(
        alphas,
        t,
        xt.shape
    )


    alpha_bar_t = extract(
        alpha_bars,
        t,
        xt.shape
    )


    # ========================================================
    # IMPLEMENTING PAPER EQUATION (125)
    # ========================================================

    mu_theta = (

        xt
        /
        torch.sqrt(
            alpha_t
        )

        -

        (
            1.0
            -
            alpha_t
        )

        /

        (
            torch.sqrt(alpha_t)

            *

            torch.sqrt(
                1.0
                -
                alpha_bar_t
            )
        )

        *

        epsilon_prediction

    )


    return mu_theta

In [ ]:
# ============================================================
# CELL 21: MODEL 3 -- NOISE LOSS
# ============================================================

def noise_prediction_loss(
    prediction,
    true_noise,
    t,
    exact_paper_weight=True
):

    squared_error = (

        prediction
        -
        true_noise

    ).pow(2).flatten(1).mean(1)


    if not exact_paper_weight:

        # Common "simple DDPM loss":
        #
        # ||epsilon - epsilon_theta||^2

        return squared_error.mean()


    alpha_t = extract(
        alphas,
        t,
        prediction.shape
    ).reshape(
        prediction.shape[0]
    )


    alpha_bar_t = extract(
        alpha_bars,
        t,
        prediction.shape
    ).reshape(
        prediction.shape[0]
    )


    sigma_q_sq = extract(
        posterior_variance,
        t,
        prediction.shape
    ).reshape(
        prediction.shape[0]
    )


    # ========================================================
    # IMPLEMENTING PAPER EQUATION (130)
    #
    #       1
    # ---------------
    # 2 sigma_q^2(t)
    #
    # *
    #
    #       (1-alpha_t)^2
    # ---------------------------
    # (1-alpha_bar_t) alpha_t
    #
    # *
    #
    # ||epsilon - epsilon_hat||^2
    # ========================================================

    weight = (

        1.0

        /

        (
            2.0
            *
            sigma_q_sq
        )

        *

        (
            1.0
            -
            alpha_t
        ).pow(2)

        /

        (

            (
                1.0
                -
                alpha_bar_t
            )

            *

            alpha_t

        )

    )


    return (
        weight
        *
        squared_error
    ).mean()

In [ ]:
# ============================================================
# CELL 22: MODEL 4 -- GROUND-TRUTH SCORE TARGET
# ============================================================

def true_score_from_noise(
    noise,
    t,
    x_shape
):
    """
    ----------------------------------------------------------
    PAPER EQUATION (151)
    ----------------------------------------------------------

                   -epsilon
        score = ---------------------
                sqrt(1-alpha_bar_t)

    where:

        score = grad_{x_t} log p(x_t)

    ----------------------------------------------------------
    """

    alpha_bar_t = extract(
        alpha_bars,
        t,
        x_shape
    )


    # ========================================================
    # IMPLEMENT PAPER EQUATION (151)
    # ========================================================

    score = (

        -noise

        /

        torch.sqrt(
            1.0
            -
            alpha_bar_t
        )

    )


    return score

In [ ]:
# ============================================================
# CELL 23: SCORE -> x_0
# ============================================================

def score_to_x0(
    xt,
    score_prediction,
    t
):
    """
    ----------------------------------------------------------
    PAPER EQUATION (133)
    ----------------------------------------------------------

                x_t
                  +
        (1-alpha_bar_t) score

        x_0 = -----------------------
                sqrt(alpha_bar_t)
    ----------------------------------------------------------
    """

    alpha_bar_t = extract(
        alpha_bars,
        t,
        xt.shape
    )


    # ========================================================
    # IMPLEMENT PAPER EQUATION (133)
    # ========================================================

    x0_prediction = (

        xt

        +

        (
            1.0
            -
            alpha_bar_t
        )

        *

        score_prediction

    ) / torch.sqrt(
        alpha_bar_t
    )


    return x0_prediction

In [ ]:
# ============================================================
# CELL 24: MODEL 4 -- SCORE -> mu_theta
# ============================================================

def score_prediction_to_mean(
    xt,
    score_prediction,
    t
):
    """
    ----------------------------------------------------------
    PAPER EQUATION (143)
    ----------------------------------------------------------

        mu_theta(x_t,t)

              1
        = ----------- x_t
          sqrt(alpha_t)

              +

          1-alpha_t
        ---------------
        sqrt(alpha_t)

          * s_theta(x_t,t)
    ----------------------------------------------------------
    """

    alpha_t = extract(
        alphas,
        t,
        xt.shape
    )


    # ========================================================
    # IMPLEMENT PAPER EQUATION (143)
    # ========================================================

    mu_theta = (

        xt
        /
        torch.sqrt(
            alpha_t
        )

        +

        (
            1.0
            -
            alpha_t
        )

        /

        torch.sqrt(
            alpha_t
        )

        *

        score_prediction

    )


    return mu_theta

In [ ]:
# ============================================================
# CELL 25: MODEL 4 -- SCORE LOSS
# ============================================================

def score_prediction_loss(
    prediction,
    true_score,
    t,
    exact_paper_weight=True
):

    squared_error = (

        prediction
        -
        true_score

    ).pow(2).flatten(1).mean(1)


    if not exact_paper_weight:

        return squared_error.mean()


    alpha_t = extract(
        alphas,
        t,
        prediction.shape
    ).reshape(
        prediction.shape[0]
    )


    sigma_q_sq = extract(
        posterior_variance,
        t,
        prediction.shape
    ).reshape(
        prediction.shape[0]
    )


    # ========================================================
    # IMPLEMENTING PAPER EQUATION (148)
    #
    #       1
    # ----------------
    # 2 sigma_q^2(t)
    #
    # *
    #
    # (1-alpha_t)^2
    # ----------------
    #     alpha_t
    #
    # *
    #
    # ||s_theta - true_score||^2
    # ========================================================

    weight = (

        1.0

        /

        (
            2.0
            *
            sigma_q_sq
        )

        *

        (
            1.0
            -
            alpha_t
        ).pow(2)

        /

        alpha_t

    )


    return (
        weight
        *
        squared_error
    ).mean()

In [ ]:
# ============================================================
# CELL 26: VERIFY ALL PARAMETERIZATIONS GIVE SAME mu_q
# ============================================================

x0_batch, _ = next(iter(train_loader))

x0_batch = x0_batch[:8].to(device)


# Avoid paper t=1 for this demonstration.
t = torch.randint(
    low=1,          # Python index 1 = paper t=2
    high=T,
    size=(8,),
    device=device
)


# ------------------------------------------------------------
# Paper Eq. (69)
# ------------------------------------------------------------

xt, epsilon = q_sample(
    x0_batch,
    t
)


# ------------------------------------------------------------
# DIRECT POSTERIOR MEAN
#
# Paper Eq. (93)
# ------------------------------------------------------------

mu_from_mean_equation = true_posterior_mean(
    x0_batch,
    xt,
    t
)


# ------------------------------------------------------------
# x_0 PARAMETERIZATION
#
# Use perfect prediction:
#
# x_hat_theta = x_0
#
# Paper Eq. (94)
# ------------------------------------------------------------

mu_from_x0 = x0_prediction_to_mean(
    xt,
    x0_batch,
    t
)


# ------------------------------------------------------------
# NOISE PARAMETERIZATION
#
# Use perfect prediction:
#
# epsilon_hat_theta = epsilon
#
# Paper Eq. (125)
# ------------------------------------------------------------

mu_from_noise = epsilon_prediction_to_mean(
    xt,
    epsilon,
    t
)


# ------------------------------------------------------------
# SCORE PARAMETERIZATION
#
# First obtain score using Paper Eq. (151).
# ------------------------------------------------------------

true_score = true_score_from_noise(
    epsilon,
    t,
    xt.shape
)


# Then use Paper Eq. (143).

mu_from_score = score_prediction_to_mean(
    xt,
    true_score,
    t
)


print(
    "Max |mu_q - mu_x0|:",
    (
        mu_from_mean_equation
        -
        mu_from_x0
    ).abs().max().item()
)


print(
    "Max |mu_q - mu_epsilon|:",
    (
        mu_from_mean_equation
        -
        mu_from_noise
    ).abs().max().item()
)


print(
    "Max |mu_q - mu_score|:",
    (
        mu_from_mean_equation
        -
        mu_from_score
    ).abs().max().item()
)

In [ ]:
# ============================================================
# CELL 27: TRAIN ONE EPOCH
# ============================================================

def train_one_epoch(
    model,
    loader,
    optimizer,
    prediction_type,
    exact_paper_weight=True
):

    model.train()

    total_loss = 0.0

    number_batches = 0


    for x0, _ in loader:

        x0 = x0.to(device)

        batch_size = x0.shape[0]


        # ----------------------------------------------------
        # SAMPLE TIMESTEP
        # ----------------------------------------------------
        #
        # The paper's denoising objective sums over:
        #
        #     t = 2,...,T
        #
        # Paper Eq. (58), Eq. (100).
        #
        # Because sigma_q^2(1)=0, the weighted loss at t=1
        # cannot use Eq. (92)/(99)/(130)/(148) directly.
        #
        # Our Python index:
        #
        #     1 -> paper timestep 2.
        # ----------------------------------------------------

        if exact_paper_weight:

            t = torch.randint(
                low=1,
                high=T,
                size=(batch_size,),
                device=device
            )

        else:

            # Simple MSE version can include all t.
            t = torch.randint(
                low=0,
                high=T,
                size=(batch_size,),
                device=device
            )


        # ----------------------------------------------------
        # epsilon ~ N(0,I)
        # ----------------------------------------------------

        epsilon = torch.randn_like(
            x0
        )


        # ----------------------------------------------------
        # PAPER EQUATION (69)
        #
        # x_t =
        #
        # sqrt(alpha_bar_t)x_0
        #
        # +
        #
        # sqrt(1-alpha_bar_t)epsilon
        # ----------------------------------------------------

        xt, epsilon = q_sample(
            x0,
            t,
            noise=epsilon
        )


        # ----------------------------------------------------
        # Neural network receives:
        #
        #     x_t, t
        #
        # Output interpretation depends on model.
        # ----------------------------------------------------

        prediction = model(
            xt,
            t
        )


        # ====================================================
        # MODEL 1: MEAN PREDICTION
        # ====================================================

        if prediction_type == "mean":

            # PAPER EQ. (93)
            target = target_mean(
                x0,
                xt,
                t
            )


            # PAPER EQ. (92)
            loss = mean_prediction_loss(
                prediction,
                target,
                t,
                exact_paper_weight
            )


        # ====================================================
        # MODEL 2: ORIGINAL SAMPLE x_0 PREDICTION
        # ====================================================

        elif prediction_type == "x0":

            # Target is simply the original clean sample:
            #
            # x_hat_theta(x_t,t) approx x_0

            target = x0


            # PAPER EQ. (99)
            loss = x0_prediction_loss(
                prediction,
                target,
                t,
                exact_paper_weight
            )


        # ====================================================
        # MODEL 3: SOURCE NOISE PREDICTION
        # ====================================================

        elif prediction_type == "noise":

            # epsilon was exactly the Gaussian noise used
            # in PAPER EQ. (69).
            #
            # Target:
            #
            # epsilon_hat_theta(x_t,t) approx epsilon_0

            target = epsilon


            # PAPER EQ. (130)
            loss = noise_prediction_loss(
                prediction,
                target,
                t,
                exact_paper_weight
            )


        # ====================================================
        # MODEL 4: SCORE PREDICTION
        # ====================================================

        elif prediction_type == "score":

            # PAPER EQ. (151)
            #
            # grad log p(x_t)
            #
            # =
            #
            # -epsilon / sqrt(1-alpha_bar_t)

            target = true_score_from_noise(
                epsilon,
                t,
                xt.shape
            )


            # PAPER EQ. (148)
            loss = score_prediction_loss(
                prediction,
                target,
                t,
                exact_paper_weight
            )


        else:

            raise ValueError(
                "prediction_type must be "
                "'mean', 'x0', 'noise', or 'score'"
            )


        # ----------------------------------------------------
        # OPTIMIZATION
        # ----------------------------------------------------

        optimizer.zero_grad()

        loss.backward()

        # Gradient clipping is not part of the paper's
        # mathematical derivation; it is simply a numerical
        # stabilization measure.

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()


        total_loss += loss.item()

        number_batches += 1


    return (
        total_loss
        /
        number_batches
    )

In [ ]:
# ============================================================
# CELL 28: TRAIN A COMPLETE MODEL
# ============================================================

def train_diffusion_model(
    prediction_type,
    epochs=10,
    learning_rate=2e-4,
    exact_paper_weight=True
):

    print(
        "\n========================================"
    )

    print(
        "Training:",
        prediction_type
    )

    print(
        "========================================"
    )


    model = MNISTDiffusionUNet().to(
        device
    )


    optimizer = optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=1e-4
    )


    history = []


    for epoch in range(
        1,
        epochs + 1
    ):

        loss = train_one_epoch(

            model=model,

            loader=train_loader,

            optimizer=optimizer,

            prediction_type=prediction_type,

            exact_paper_weight=exact_paper_weight

        )


        history.append(
            loss
        )


        print(
            f"{prediction_type:>6s} | "
            f"Epoch {epoch:02d}/{epochs} | "
            f"Loss = {loss:.6f}"
        )


    return model, history

In [ ]:
# ============================================================
# CELL 29: TRAIN MODEL 1 -- MEAN PARAMETERIZATION
# ============================================================

EPOCHS = 30

EXACT_PAPER_WEIGHT = True


mean_model, mean_history = train_diffusion_model(

    prediction_type="mean",

    epochs=EPOCHS,

    exact_paper_weight=EXACT_PAPER_WEIGHT

)

In [ ]:
# ============================================================
# CELL 30: TRAIN MODEL 2 -- x_0 / SOURCE SAMPLE
# ============================================================

x0_model, x0_history = train_diffusion_model(

    prediction_type="x0",

    epochs=EPOCHS,

    exact_paper_weight=EXACT_PAPER_WEIGHT

)

In [ ]:
# ============================================================
# CELL 31: TRAIN MODEL 3 -- NOISE PARAMETERIZATION
# ============================================================

noise_model, noise_history = train_diffusion_model(

    prediction_type="noise",

    epochs=EPOCHS,

    exact_paper_weight=EXACT_PAPER_WEIGHT

)

In [ ]:
# ============================================================
# CELL 32: TRAIN MODEL 4 -- SCORE PARAMETERIZATION
# ============================================================

score_model, score_history = train_diffusion_model(

    prediction_type="score",

    epochs=EPOCHS,

    exact_paper_weight=EXACT_PAPER_WEIGHT

)

In [ ]:
# ============================================================
# CELL 33: TRAINING CURVES
# ============================================================

plt.figure(figsize=(8, 5))

plt.plot(
    mean_history,
    label="Mean prediction"
)

plt.plot(
    x0_history,
    label="x0 prediction"
)

plt.plot(
    noise_history,
    label="Noise prediction"
)

plt.plot(
    score_history,
    label="Score prediction"
)

plt.xlabel("Epoch")
plt.ylabel("Training objective")

plt.title(
    "Four Diffusion Parameterizations"
)

plt.legend()

plt.show()

In [ ]:
# ============================================================
# CELL 34: MODEL OUTPUT -> REVERSE MEAN
# ============================================================

def model_output_to_mean(
    model,
    xt,
    t,
    prediction_type
):

    output = model(
        xt,
        t
    )


    # ========================================================
    # MODEL 1
    #
    # Network DIRECTLY predicts mu_theta.
    #
    # Related to PAPER EQ. (92)
    # ========================================================

    if prediction_type == "mean":

        mu_theta = output


    # ========================================================
    # MODEL 2
    #
    # Network predicts x_0.
    #
    # Convert x_hat_theta -> mu_theta using PAPER EQ. (94).
    # ========================================================

    elif prediction_type == "x0":

        x0_prediction = output.clamp(
            -1,
            1
        )

        mu_theta = x0_prediction_to_mean(
            xt,
            x0_prediction,
            t
        )


    # ========================================================
    # MODEL 3
    #
    # Network predicts epsilon.
    #
    # Convert epsilon_hat -> mu_theta using PAPER EQ. (125).
    # ========================================================

    elif prediction_type == "noise":

        epsilon_prediction = output

        mu_theta = epsilon_prediction_to_mean(
            xt,
            epsilon_prediction,
            t
        )


    # ========================================================
    # MODEL 4
    #
    # Network predicts score.
    #
    # Convert score -> mu_theta using PAPER EQ. (143).
    # ========================================================

    elif prediction_type == "score":

        score_prediction = output

        mu_theta = score_prediction_to_mean(
            xt,
            score_prediction,
            t
        )


    else:

        raise ValueError(
            "Unknown prediction type"
        )


    return mu_theta

In [ ]:
# ============================================================
# CELL 35: ONE REVERSE DIFFUSION STEP
# ============================================================

@torch.no_grad()
def p_sample(
    model,
    xt,
    t,
    prediction_type
):
    """
    Sample:

        x_{t-1} ~ p_theta(x_{t-1}|x_t)

    We model:

        p_theta(x_{t-1}|x_t)

        =
        N(
            mu_theta(x_t,t),
            sigma_q^2(t) I
          )

    The variance is PAPER EQ. (85).

    The mean depends on parameterization:

        mean  : direct mu_theta
        x0    : PAPER EQ. (94)
        noise : PAPER EQ. (125)
        score : PAPER EQ. (143)
    """


    # --------------------------------------------------------
    # Compute reverse mean.
    # --------------------------------------------------------

    mu_theta = model_output_to_mean(
        model,
        xt,
        t,
        prediction_type
    )


    # --------------------------------------------------------
    # PAPER EQUATION (85)
    #
    # sigma_q^2(t)
    # --------------------------------------------------------

    variance = extract(
        posterior_variance,
        t,
        xt.shape
    )


    # --------------------------------------------------------
    # At paper timestep t=1:
    #
    # x_0 = mu_theta
    #
    # No additional noise is required.
    #
    # Python t=0 corresponds to paper t=1.
    # --------------------------------------------------------

    noise = torch.randn_like(
        xt
    )


    nonzero_mask = (

        (t != 0)
        .float()
        .reshape(
            xt.shape[0],
            1,
            1,
            1
        )

    )


    # --------------------------------------------------------
    # Reverse Gaussian sampling:
    #
    # x_{t-1}
    #
    # =
    #
    # mu_theta
    #
    # +
    #
    # sigma_q(t) z
    #
    # z ~ N(0,I)
    # --------------------------------------------------------

    x_previous = (

        mu_theta

        +

        nonzero_mask

        *

        torch.sqrt(
            variance.clamp(
                min=0.0
            )
        )

        *

        noise

    )


    return x_previous

In [ ]:
# ============================================================
# CELL 36: FULL REVERSE SAMPLING
# ============================================================

@torch.no_grad()
def generate_samples(
    model,
    prediction_type,
    num_samples=25
):

    model.eval()


    # --------------------------------------------------------
    # PAPER EQ. (33)
    #
    # p(x_T) = N(0,I)
    #
    # Start completely from Gaussian noise.
    # --------------------------------------------------------

    xt = torch.randn(
        num_samples,
        1,
        28,
        28,
        device=device
    )


    # --------------------------------------------------------
    # Reverse:
    #
    # T, T-1, ..., 2, 1
    #
    # Python:
    #
    # T-1, ..., 1, 0
    # --------------------------------------------------------

    for time_index in reversed(
        range(T)
    ):

        t = torch.full(
            (num_samples,),
            time_index,
            device=device,
            dtype=torch.long
        )


        xt = p_sample(
            model,
            xt,
            t,
            prediction_type
        )


    # Return x_0.
    return xt.clamp(
        -1,
        1
    )

In [ ]:
# ============================================================
# CELL 37: GENERATE -- MEAN MODEL
# ============================================================

mean_samples = generate_samples(

    mean_model,

    prediction_type="mean",

    num_samples=25

)

In [ ]:
# ============================================================
# CELL 38: GENERATE -- SOURCE SAMPLE MODEL
# ============================================================

x0_samples = generate_samples(

    x0_model,

    prediction_type="x0",

    num_samples=25

)

In [ ]:
# ============================================================
# CELL 39: GENERATE -- NOISE MODEL
# ============================================================

noise_samples = generate_samples(

    noise_model,

    prediction_type="noise",

    num_samples=25

)

In [ ]:
# ============================================================
# CELL 40: GENERATE -- SCORE MODEL
# ============================================================

score_samples = generate_samples(

    score_model,

    prediction_type="score",

    num_samples=25

)

In [ ]:
# ============================================================
# CELL 41: DISPLAY GENERATED IMAGES
# ============================================================

def show_generated(
    samples,
    title
):

    # [-1,1] -> [0,1]

    samples = (
        samples.detach().cpu()
        +
        1
    ) / 2


    plt.figure(
        figsize=(8, 8)
    )


    for i in range(
        min(25, len(samples))
    ):

        plt.subplot(
            5,
            5,
            i + 1
        )

        plt.imshow(
            samples[
                i, 0
            ],
            cmap="gray"
        )

        plt.axis("off")


    plt.suptitle(
        title
    )

    plt.tight_layout()

    plt.show()

In [ ]:
show_generated(

    mean_samples,

    "Model 1: Direct Reverse-Mean Prediction"
)

In [ ]:
show_generated(

    x0_samples,

    "Model 2: Original Sample x0 Prediction"
)

In [ ]:
show_generated(

    noise_samples,

    "Model 3: Source Noise Prediction"
)

In [ ]:
show_generated(

    score_samples,

    "Model 4: Score Prediction"
)

In [ ]:
# ============================================================
# CELL 46: SAME x_T FOR ALL FOUR MODELS
# ============================================================

@torch.no_grad()
def generate_from_fixed_noise(
    model,
    initial_noise,
    prediction_type
):

    model.eval()

    xt = initial_noise.clone()


    for time_index in reversed(
        range(T)
    ):

        t = torch.full(

            (
                xt.shape[0],
            ),

            time_index,

            device=device,

            dtype=torch.long
        )


        xt = p_sample(

            model,

            xt,

            t,

            prediction_type
        )


    return xt.clamp(
        -1,
        1
    )

In [ ]:
# ============================================================
# CELL 47: FIXED INITIAL GAUSSIAN NOISE
# ============================================================

NUM_COMPARE = 10


# PAPER EQ. (33)
#
# x_T ~ N(0,I)

fixed_xT = torch.randn(

    NUM_COMPARE,

    1,

    28,

    28,

    device=device

)

In [ ]:
# ============================================================
# CELL 48: GENERATE USING SAME x_T
# ============================================================

comparison_mean = generate_from_fixed_noise(

    mean_model,

    fixed_xT,

    "mean"
)


comparison_x0 = generate_from_fixed_noise(

    x0_model,

    fixed_xT,

    "x0"
)


comparison_noise = generate_from_fixed_noise(

    noise_model,

    fixed_xT,

    "noise"
)


comparison_score = generate_from_fixed_noise(

    score_model,

    fixed_xT,

    "score"
)

In [ ]:
# ============================================================
# CELL 49: VISUAL COMPARISON
# ============================================================

rows = [

    ("Mean", comparison_mean),

    ("x0", comparison_x0),

    ("Noise", comparison_noise),

    ("Score", comparison_score)

]


plt.figure(
    figsize=(15, 7)
)


for row_index, (
    name,
    samples
) in enumerate(rows):


    display_samples = (

        samples.cpu()
        +
        1

    ) / 2


    for i in range(NUM_COMPARE):

        plt.subplot(

            4,

            NUM_COMPARE,

            row_index
            *
            NUM_COMPARE

            +
            i
            +
            1
        )


        plt.imshow(

            display_samples[
                i, 0
            ],

            cmap="gray"

        )


        plt.axis("off")


        if i == 0:

            plt.ylabel(
                name
            )


plt.tight_layout()

plt.show()

In [ ]:
# ============================================================
# CELL 50: WHAT EXACTLY DOES EACH MODEL PREDICT?
# ============================================================

x0_example, _ = next(
    iter(train_loader)
)

x0_example = x0_example[
    0:1
].to(device)


paper_t = 150

t = torch.tensor(
    [paper_t - 1],
    device=device
)


# ------------------------------------------------------------
# PAPER EQ. (69)
# ------------------------------------------------------------

xt_example, epsilon_example = q_sample(
    x0_example,
    t
)


# ------------------------------------------------------------
# MODEL 1 TARGET
#
# PAPER EQ. (93)
# ------------------------------------------------------------

mean_target = true_posterior_mean(

    x0_example,

    xt_example,

    t
)


# ------------------------------------------------------------
# MODEL 2 TARGET
#
# x_0 itself.
# ------------------------------------------------------------

x0_target = x0_example


# ------------------------------------------------------------
# MODEL 3 TARGET
#
# epsilon used in PAPER EQ. (69).
# ------------------------------------------------------------

noise_target = epsilon_example


# ------------------------------------------------------------
# MODEL 4 TARGET
#
# PAPER EQ. (151)
# ------------------------------------------------------------

score_target = true_score_from_noise(

    epsilon_example,

    t,

    xt_example.shape

)

In [ ]:
# ============================================================
# CELL 51: VISUALIZE TARGETS
# ============================================================

fig = plt.figure(
    figsize=(15, 3)
)


# Clean image
plt.subplot(1, 5, 1)

plt.imshow(
    x0_example[
        0, 0
    ].cpu(),
    cmap="gray"
)

plt.title(
    r"$x_0$"
)

plt.axis("off")


# Noisy x_t
plt.subplot(1, 5, 2)

plt.imshow(
    xt_example[
        0, 0
    ].cpu(),
    cmap="gray"
)

plt.title(
    rf"$x_t$, t={paper_t}"
)

plt.axis("off")


# Posterior mean target
plt.subplot(1, 5, 3)

plt.imshow(
    mean_target[
        0, 0
    ].cpu(),
    cmap="gray"
)

plt.title(
    r"$\mu_q$ target"
)

plt.axis("off")


# Noise target
plt.subplot(1, 5, 4)

plt.imshow(
    noise_target[
        0, 0
    ].cpu(),
    cmap="gray"
)

plt.title(
    r"$\epsilon$ target"
)

plt.axis("off")


# Score target
plt.subplot(1, 5, 5)

plt.imshow(
    score_target[
        0, 0
    ].cpu(),
    cmap="gray"
)

plt.title(
    "Score target"
)

plt.axis("off")


plt.tight_layout()

plt.show()

In [ ]:
# ============================================================
# CELL 52: VERIFY NOISE <-> SCORE RELATION
# ============================================================

alpha_bar_t = extract(

    alpha_bars,

    t,

    xt_example.shape

)


# ------------------------------------------------------------
# PAPER EQ. (151), REARRANGED:
#
# epsilon
#
# =
#
# -sqrt(1-alpha_bar_t)
#
# * score
# ------------------------------------------------------------

epsilon_recovered = (

    -torch.sqrt(
        1.0
        -
        alpha_bar_t
    )

    *

    score_target

)


print(
    "Maximum error:",
    (
        epsilon_example
        -
        epsilon_recovered
    ).abs().max().item()
)

In [ ]:
# ============================================================
# CELL 53: CONVERSION BETWEEN PARAMETERIZATIONS
# ============================================================

def x0_to_epsilon(
    xt,
    x0_prediction,
    t
):
    """
    Rearranged PAPER EQ. (69):

        epsilon

        =
        [x_t - sqrt(alpha_bar_t)x_0]
        --------------------------------
            sqrt(1-alpha_bar_t)
    """

    alpha_bar_t = extract(
        alpha_bars,
        t,
        xt.shape
    )


    epsilon = (

        xt

        -

        torch.sqrt(
            alpha_bar_t
        )

        *

        x0_prediction

    ) / torch.sqrt(

        1.0
        -
        alpha_bar_t

    )


    return epsilon



def epsilon_to_score(
    epsilon_prediction,
    t,
    x_shape
):
    """
    PAPER EQ. (151):

        score

        =
        -epsilon
        ---------------------
        sqrt(1-alpha_bar_t)
    """

    alpha_bar_t = extract(
        alpha_bars,
        t,
        x_shape
    )


    score = (

        -epsilon_prediction

        /

        torch.sqrt(
            1.0
            -
            alpha_bar_t
        )

    )


    return score



def score_to_epsilon(
    score_prediction,
    t,
    x_shape
):
    """
    PAPER EQ. (151), rearranged:

        epsilon

        =
        -sqrt(1-alpha_bar_t)
        * score
    """

    alpha_bar_t = extract(
        alpha_bars,
        t,
        x_shape
    )


    epsilon = (

        -torch.sqrt(
            1.0
            -
            alpha_bar_t
        )

        *

        score_prediction

    )


    return epsilon